# YOLO11 Eğitim Örneği / YOLO11 Training Example

Bu notebook, YOLO11 modelinin özel veri seti üzerinde nasıl eğitileceğini gösterir.
This notebook demonstrates how to fine-tune YOLO11 on a custom dataset.

## Eğitim Süreci / Training Process:
1. Model seçimi ve yükleme
2. Veri seti yapılandırması
3. Eğitim parametrelerinin ayarlanması
4. Model eğitimi
5. Model değerlendirmesi
6. Model kaydetme ve inference

In [ ]:
# Import required libraries
import os
import sys
from pathlib import Path

# Add project root to path
project_root = Path('../').resolve()
sys.path.insert(0, str(project_root))

import yaml
import torch
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Image

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## Ultralytics YOLO11 Yükleme / Installing Ultralytics YOLO11

In [ ]:
# Install/upgrade ultralytics
# !pip install ultralytics>=8.0.0

from ultralytics import YOLO

print("Ultralytics imported successfully!")

# Check YOLO version
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

## Veri Seti Yapılandırması / Dataset Configuration

In [ ]:
# Dataset configuration
DATASET_ROOT = Path("../datasets/defect_detection")
DATASET_YAML = DATASET_ROOT / "dataset.yaml"

# Load dataset config
with open(DATASET_YAML, 'r') as f:
    dataset_config = yaml.safe_load(f)

print("=" * 50)
print("DATASET CONFIGURATION")
print("=" * 50)
print(f"Name: {dataset_config.get('name')}")
print(f"Path: {dataset_config.get('path')}")
print(f"Classes: {dataset_config.get('nc')}")
print(f"Class names: {dataset_config.get('names')}")

# Class names for training
CLASS_NAMES = dataset_config.get('names', {})
print(f"\nClasses to train: {list(CLASS_NAMES.values())}")

## Model Seçimi / Model Selection

In [ ]:
# YOLO11 Model Options:
# yolo11n-seg.pt  - Nano (en hızlı, en az doğru)
# yolo11s-seg.pt  - Small
# yolo11m-seg.pt  - Medium
# yolo11l-seg.pt  - Large
# yolo11x-seg.pt  - XLarge (en yavaş, en doğru)

MODEL_SIZE = "n"  # Use 'n' for nano (fastest), 's', 'm', 'l', 'x'
MODEL_NAME = f"yolo11{MODEL_SIZE}-seg.pt"

print(f"Loading model: {MODEL_NAME}")

# Load pre-trained model
model = YOLO(MODEL_NAME)

print("Model loaded successfully!")
print(f"Model task: {model.task}")
print(f"Model classes: {model.names}")

## Eğitim Yapılandırması / Training Configuration

In [ ]:
# Training parameters

TRAIN_CONFIG = {
    # Data configuration
    'data': str(DATASET_YAML.resolve()),
    
    # Model configuration
    'model': MODEL_NAME,
    'epochs': 100,           # Number of training epochs
    'imgsz': 640,           # Image size
    'batch': 16,            # Batch size (adjust based on GPU memory)
    
    # Training settings
    'optimizer': 'SGD',     # Optimizer: SGD, Adam, AdamW
    'lr0': 0.01,            # Initial learning rate
    'lrf': 0.01,            # Final learning rate factor
    'momentum': 0.937,      # SGD momentum
    'weight_decay': 0.0005, # Optimizer weight decay
    
    # Augmentation
    'hsv_h': 0.015,         # HSV-Hue augmentation
    'hsv_s': 0.7,          # HSV-Saturation
    'hsv_v': 0.4,          # HSV-Value
    'degrees': 0.0,         # Rotation
    'translate': 0.1,      # Translation
    'scale': 0.5,          # Scale
    'shear': 0.0,          # Shear
    'flipud': 0.0,         # Vertical flip
    'fliplr': 0.5,         # Horizontal flip
    'mosaic': 1.0,         # Mosaic augmentation
    'mixup': 0.0,          # MixUp augmentation
    'copy_paste': 0.0,     # Copy-paste augmentation
    
    # Other settings
    'patience': 50,         # Early stopping patience
    'save': True,          # Save checkpoints
    'save_period': 10,     # Save checkpoint every N epochs
    'cache': False,        # Cache images
    'device': '',          # Device (empty = auto-detect)
    'workers': 8,          # Number of worker threads
    'project': '../runs/segment',  # Project directory
    'name': 'exp',         # Experiment name
    'exist_ok': True,      # Overwrite existing
    'pretrained': True,   # Use pretrained weights
    'verbose': True,       # Verbose output
    'seed': 0,             # Random seed
    'deterministic': True, # Deterministic training
}

print("Training Configuration:")
print("=" * 50)
for key, value in TRAIN_CONFIG.items():
    print(f"  {key}: {value}")

## Model Eğitimi / Model Training

In [ ]:
# Start training
print("=" * 50)
print("STARTING TRAINING")
print("=" * 50)

# Note: Uncomment the following code to start training
# This is commented to prevent accidental training in notebook

# results = model.train(**TRAIN_CONFIG)

print("\nTraining command (commented out to prevent accidental training):")
print("results = model.train(**TRAIN_CONFIG)")

print("\nTo start training, uncomment the line above and run the cell.")
print("Expected training time depends on:")
print("  - Number of epochs")
print("  - Dataset size")
print("  - GPU performance")
print("  - Batch size")

## Eğitim Sonuçlarını Analiz Etme / Analyzing Training Results

In [ ]:
# Function to plot training results
def plot_training_results(results_dir):
    """
    Plot training results from results.csv
    
    Args:
        results_dir: Path to training results directory
    """
    import pandas as pd
    
    results_csv = Path(results_dir) / "results.csv"
    
    if not results_csv.exists():
        print(f"Results file not found: {results_csv}")
        return
    
    # Load results
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()  # Remove whitespace
    
    # Plot metrics
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    
    # Loss metrics
    if 'train/box_loss' in df.columns:
        axes[0, 0].plot(df['epoch'], df['train/box_loss'], label='Box Loss')
        axes[0, 0].set_title('Box Loss')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Loss')
        axes[0, 0].legend()
    
    if 'train/seg_loss' in df.columns:
        axes[0, 1].plot(df['epoch'], df['train/seg_loss'], label='Seg Loss', color='orange')
        axes[0, 1].set_title('Segmentation Loss')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Loss')
        axes[0, 1].legend()
    
    if 'train/cls_loss' in df.columns:
        axes[0, 2].plot(df['epoch'], df['train/cls_loss'], label='Cls Loss', color='green')
        axes[0, 2].set_title('Classification Loss')
        axes[0, 2].set_xlabel('Epoch')
        axes[0, 2].set_ylabel('Loss')
        axes[0, 2].legend()
    
    # Validation metrics
    if 'metrics/mAP50(B)' in df.columns:
        axes[1, 0].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@0.5', color='blue')
        axes[1, 0].set_title('mAP@0.5 (Bounding Box)')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('mAP')
        axes[1, 0].legend()
    
    if 'metrics/mAP50-95(B)' in df.columns:
        axes[1, 1].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95', color='purple')
        axes[1, 1].set_title('mAP@0.5:0.95 (Bounding Box)')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('mAP')
        axes[1, 1].legend()
    
    if 'metrics/mAP50-95(M)' in df.columns:
        axes[1, 2].plot(df['epoch'], df['metrics/mAP50-95(M)'], label='mAP@0.5:0.95', color='red')
        axes[1, 2].set_title('mAP@0.5:0.95 (Mask)')
        axes[1, 2].set_xlabel('Epoch')
        axes[1, 2].set_ylabel('mAP')
        axes[1, 2].legend()
    
    plt.tight_layout()
    plt.show()

print("Training results plotting function defined.")
print("\nUsage:")
print("  plot_training_results('../runs/segment/exp')")

## Model Değerlendirmesi / Model Evaluation

In [ ]:
# Evaluate trained model

def evaluate_model(model_path, data_yaml, imgsz=640):
    """
    Evaluate trained model on validation set.
    
    Args:
        model_path: Path to trained model weights
        data_yaml: Path to dataset YAML
        imgsz: Image size
    
    Returns:
        Evaluation results
    """
    model = YOLO(model_path)
    
    results = model.val(
        data=data_yaml,
        imgsz=imgsz,
        split='val',
        verbose=True
    )
    
    return results

print("Model evaluation function defined.")

# Example usage (commented out)
# results = evaluate_model('../runs/segment/exp/weights/best.pt', str(DATASET_YAML))

print("\nUsage:")
print("  results = evaluate_model('../runs/segment/exp/weights/best.pt', str(DATASET_YAML))")
print("\nExpected evaluation metrics:")
print("  - mAP@0.5 (mean Average Precision at IoU=0.5)")
print("  - mAP@0.5:0.95 (mean Average Precision at IoU=0.5:0.95)")
print("  - Precision")
print("  - Recall")

## Model Çıkarımı / Model Inference

In [ ]:
# Run inference with trained model

def run_inference(model_path, image_source, conf=0.25, iou=0.45):
    """
    Run inference on images or video.
    
    Args:
        model_path: Path to trained model
        image_source: Image path, directory, or video
        conf: Confidence threshold
        iou: IoU threshold for NMS
    
    Returns:
        Inference results
    """
    model = YOLO(model_path)
    
    results = model.predict(
        source=image_source,
        conf=conf,
        iou=iou,
        save=True,
        save_txt=True,
        save_conf=True
    )
    
    return results

print("Inference function defined.")

# Example usage
# results = run_inference('../runs/segment/exp/weights/best.pt', '../datasets/defect_detection/val/images')

print("\nUsage:")
print("  # Single image")
print("  results = run_inference('path/to/model.pt', 'image.jpg')")
print("  ")
print("  # Folder")
print("  results = run_inference('path/to/model.pt', 'folder/')")
print("  ")
print("  # Video")
print("  results = run_inference('path/to/model.pt', 'video.mp4')")

## Model Export / Model Export

In [ ]:
# Export trained model to different formats

def export_model(model_path, format='onnx', imgsz=640):
    """
    Export trained model to different formats.
    
    Args:
        model_path: Path to trained model
        format: Export format (onnx, torchscript, coreml, tflite, etc.)
        imgsz: Image size
    
    Returns:
        Export path
    """
    model = YOLO(model_path)
    
    export_path = model.export(
        format=format,
        imgsz=imgsz,
        dynamic=False,
        simplify=True
    )
    
    return export_path

print("Model export function defined.")

# Available export formats
print("\nAvailable export formats:")
print("  - onnx: ONNX format (cross-platform)")
print("  - torchscript: TorchScript format (PyTorch)")
print("  - coreml: CoreML format (iOS)")
print("  - tflite: TensorFlow Lite (mobile)")
print("  - edgetpu: Edge TPU (Google Coral)")
print("  - paddlepaddle: PaddlePaddle format")
print("  - ncnn: NCNN format (mobile)")

# Example usage
# export_path = export_model('../runs/segment/exp/weights/best.pt', 'onnx')
# print(f"Exported to: {export_path}")

## İleri Düzey Eğitim İpuçları / Advanced Training Tips

In [ ]:
# Advanced training tips

print("ADVANCED TRAINING TIPS:")
print("=" * 50)

print("\n1. HYPERPARAMETER OPTIMIZATION:")
print("   - Use optuna or raytune for hyperparameter search")
print("   - Key parameters: lr0, lrf, momentum, weight_decay")

print("\n2. MIXED PRECISION TRAINING:")
print("   - Use 'amp=True' in training config")
print("   - Reduces memory usage, speeds up training")

print("\n3. EARLY STOPPING:")
print("   - Set 'patience' parameter")
print("   - Stops training if no improvement for N epochs")

print("\n4. TRANSFER LEARNING:")
print("   - Start from a pre-trained model")
print("   - Fine-tune on your custom dataset")
print("   - Use 'freeze' parameter to freeze layers")

print("\n5. DATA AUGMENTATION:")
print("   - Adjust augmentation parameters in config")
print("   - Use mosaic, mixup for better generalization")
print("   - Consider Test-Time Augmentation (TTA)")

print("\n6. MODEL ENSEMBLING:")
print("   - Combine multiple models for better accuracy")
print("   - Use WBF (Weighted Box Fusion) for post-processing")

print("\n7. GPU OPTIMIZATION:")
print("   - Adjust batch size based on GPU memory")
print("   - Use gradient accumulation for larger effective batch")
print("   - Enable mixed precision training (amp)")

## Özet / Summary

Bu notebook şunları gösterdi:
- YOLO11 modelinin yapılandırılması
- Eğitim parametrelerinin ayarlanması
- Model eğitimi
- Eğitim sonuçlarının analizi
- Model değerlendirmesi
- Model export etme
- İleri düzey eğitim ipuçları
---
This notebook demonstrated:
- Configuring YOLO11 model
- Setting training parameters
- Training the model
- Analyzing training results
- Evaluating the model
- Exporting the model
- Advanced training tips